# Repository-Aware Retrieval and RAG

This notebook implements semantic retrieval over historical BugsInPy examples and uses retrieved bug context to augment LLM-based bug diagnosis and fix recommendations.

In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
bugs_df = pd.read_csv(
    "../data/processed/bug_dataset_ml.csv"
)

historical_embeddings = np.load(
    "../data/processed/minilm_patch_embeddings.npy"
)

print(bugs_df.shape)
print(historical_embeddings.shape)

(484, 26)
(484, 384)


In [3]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

tokenizer = embedding_model.tokenizer

In [4]:
def embed_patch(
    text,
    model,
    tokenizer,
    chunk_size=240,
    overlap=40
):
    text = str(text)

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False,
        verbose=False
    )

    chunk_embeddings = []

    step = chunk_size - overlap

    for start in range(0, len(token_ids), step):
        end = start + chunk_size

        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True
        )

        embedding = model.encode(
            chunk_text,
            convert_to_numpy=True
        )

        chunk_embeddings.append(embedding)

        if end >= len(token_ids):
            break

    return np.mean(
        chunk_embeddings,
        axis=0
    )

In [5]:
def retrieve_similar_bugs(
    query_patch,
    bugs_df,
    historical_embeddings,
    model,
    tokenizer,
    top_k=5
):
    query_embedding = embed_patch(
        query_patch,
        model,
        tokenizer
    )

    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        historical_embeddings
    )[0]

    top_indices = similarities.argsort()[::-1][:top_k]

    results = bugs_df.iloc[top_indices].copy()

    results["similarity_score"] = similarities[top_indices]

    return results

In [6]:
test_patch = bugs_df["patch"].iloc[0]

results = retrieve_similar_bugs(
    test_patch,
    bugs_df,
    historical_embeddings,
    embedding_model,
    tokenizer,
    top_k=5
)

In [7]:
results[
    [
        "project",
        "bug_id",
        "bug_category",
        "modified_files",
        "similarity_score"
    ]
]

,project,bug_id,bug_category,modified_files,similarity_score
0,keras,32,API and Configuration,['keras/callbacks.py'],1.000000
20,keras,21,API and Configuration,['keras/callbacks.py'],0.741499
10,keras,29,Program Logic and State Management,['keras/engine/training.py'],0.632105
31,keras,4,API and Configuration,['keras/optimizers.py'],0.574832
29,keras,41,Program Logic and State Management,"['keras/utils/data_utils.py', 'tests/test_mult...",0.554467


In [8]:
def retrieve_similar_bugs(
    query_patch,
    bugs_df,
    historical_embeddings,
    model,
    tokenizer,
    top_k=5,
    exclude_index=None
):
    query_embedding = embed_patch(
        query_patch,
        model,
        tokenizer
    )

    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        historical_embeddings
    )[0]

    # Exclude the query bug itself during evaluation
    if exclude_index is not None:
        similarities[exclude_index] = -1

    top_indices = similarities.argsort()[::-1][:top_k]

    results = bugs_df.iloc[top_indices].copy()
    results["similarity_score"] = similarities[top_indices]

    return results

In [9]:
query_index = 0

results = retrieve_similar_bugs(
    bugs_df["patch"].iloc[query_index],
    bugs_df,
    historical_embeddings,
    embedding_model,
    tokenizer,
    top_k=5,
    exclude_index=query_index
)

results[
    [
        "project",
        "bug_id",
        "bug_category",
        "modified_files",
        "similarity_score"
    ]
]

,project,bug_id,bug_category,modified_files,similarity_score
20,keras,21,API and Configuration,['keras/callbacks.py'],0.741499
10,keras,29,Program Logic and State Management,['keras/engine/training.py'],0.632105
31,keras,4,API and Configuration,['keras/optimizers.py'],0.574832
29,keras,41,Program Logic and State Management,"['keras/utils/data_utils.py', 'tests/test_mult...",0.554467
168,tornado,16,Program Logic and State Management,['tornado/gen.py'],0.547965


In [10]:
def hit_at_k(k):
    hits = 0

    for i in range(len(bugs_df)):
        query_embedding = historical_embeddings[i]

        similarities = cosine_similarity(
            query_embedding.reshape(1, -1),
            historical_embeddings
        )[0]

        # Exclude itself
        similarities[i] = -1

        top_indices = similarities.argsort()[::-1][:k]

        query_category = bugs_df.iloc[i]["bug_category"]

        retrieved_categories = bugs_df.iloc[top_indices]["bug_category"].values

        if query_category in retrieved_categories:
            hits += 1

    return hits / len(bugs_df)

In [11]:
for k in [1, 3, 5]:
    print(f"Hit@{k}: {hit_at_k(k):.3f}")

Hit@1: 0.395
Hit@3: 0.651
Hit@5: 0.773


* Hit@1 = 0.395 → the single nearest neighbor has the same category about 39.5% of the time.
* Hit@3 = 0.651 → at least one of the top 3 matches the category about 65.1% of the time.
* Hit@5 = 0.773 → at least one of the top 5 matches about 77.3% of the time.

In [12]:
def build_rag_context(retrieved_bugs):
    context_parts = []

    for rank, (_, bug) in enumerate(
        retrieved_bugs.iterrows(),
        start=1
    ):
        context_parts.append(
            f"""
HISTORICAL BUG {rank}

Project: {bug['project']}
Bug ID: {bug['bug_id']}
Bug Category: {bug['bug_category']}
Modified Files: {bug['modified_files']}

Patch:
{bug['patch']}
"""
        )

    return "\n".join(context_parts)

In [13]:
rag_context = build_rag_context(results)

print(rag_context[:3000])


HISTORICAL BUG 1

Project: keras
Bug ID: 21
Bug Category: API and Configuration
Modified Files: ['keras/callbacks.py']

Patch:
diff --git a/keras/callbacks.py b/keras/callbacks.py
index 067169f3..04826921 100644
--- a/keras/callbacks.py
+++ b/keras/callbacks.py
@@ -477,6 +477,10 @@ class EarlyStopping(Callback):
         baseline: Baseline value for the monitored quantity to reach.
             Training will stop if the model doesn't show improvement
             over the baseline.
+        restore_best_weights: whether to restore model weights from
+            the epoch with the best value of the monitored quantity.
+            If False, the model weights obtained at the last step of
+            training are used.
     """
 
     def __init__(self,
@@ -485,7 +489,8 @@ class EarlyStopping(Callback):
                  patience=0,
                  verbose=0,
                  mode='auto',
-                 baseline=None):
+                 baseline=None,
+                 restore_be

In [14]:
def build_rag_prompt(user_patch, rag_context):
    return f"""
You are a Python debugging assistant.

Analyze the user's bug patch using the retrieved historical
bug examples as supporting context.

The retrieved bugs may be similar but may not have the same
root cause. Use them only when relevant.

USER BUG PATCH:
{user_patch}

RETRIEVED HISTORICAL BUGS:
{rag_context}

Provide:

1. Likely root cause
2. Likely bug category
3. Explanation
4. Recommended debugging steps
5. Suggested fix
""".strip()

In [15]:
prompt = build_rag_prompt(
    test_patch,
    rag_context
)

print(prompt[:3000])

You are a Python debugging assistant.

Analyze the user's bug patch using the retrieved historical
bug examples as supporting context.

The retrieved bugs may be similar but may not have the same
root cause. Use them only when relevant.

USER BUG PATCH:
diff --git a/keras/callbacks.py b/keras/callbacks.py
index 8b5ec5f3..eb233de3 100644
--- a/keras/callbacks.py
+++ b/keras/callbacks.py
@@ -898,7 +898,7 @@ class ReduceLROnPlateau(Callback):
             monitored has stopped increasing; in `auto`
             mode, the direction is automatically inferred
             from the name of the monitored quantity.
-        epsilon: threshold for measuring the new optimum,
+        min_delta: threshold for measuring the new optimum,
             to only focus on significant changes.
         cooldown: number of epochs to wait before resuming
             normal operation after lr has been reduced.
@@ -906,16 +906,21 @@ class ReduceLROnPlateau(Callback):
     """
 
     def __init__(self, monito

In [18]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass(
    "Enter OpenAI API key: "
)

In [19]:
from openai import OpenAI

client = OpenAI()

In [20]:
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

API key loaded: True


In [23]:
response = client.responses.create(
    model="gpt-5.6-luna",
    input=prompt
)

generated_answer = response.output_text

print(generated_answer)

1. **Likely root cause**

There are two likely issues addressed by this patch:

- **Off-by-one error in patience handling:** `self.wait` was incremented after the learning-rate reduction check. Consequently, when `wait >= patience`, the counter was reset to zero and then immediately incremented to one. More importantly, reduction occurred one epoch later than expected.
- **Inconsistent API terminology:** `ReduceLROnPlateau` used `epsilon`, while related callbacks such as `EarlyStopping` use `min_delta`. This creates an inconsistent public API and makes configuration harder to understand.

The first issue is a program-state bug; the second is an API compatibility issue.

2. **Likely bug category**

Primarily:

- **Program Logic and State Management** — incorrect `wait` counter sequencing.
- **API and Configuration** — renaming `epsilon` to `min_delta` while retaining backward compatibility.

This is somewhat analogous to Historical Bug 1, where `EarlyStopping` standardized on `min_delta

In [28]:
import ollama

ollama_response = ollama.chat(
    model="llama3:8b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

ollama_answer = ollama_response["message"]["content"]

print(ollama_answer)

Based on the provided bug patch, I'll analyze the likely root cause, bug category, explanation, recommended debugging steps, and suggested fix.

**Likely Root Cause:**
The bug likely arises from the changes made in the `ReduceLROnPlateau` callback class. Specifically, the modification of the `min_delta` attribute and the addition of a new attribute `iterations` in the `get_updates` method.

**Likely Bug Category:**
The bug category is likely "Program Logic and State Management" because it involves changes to the internal state and logic of the `ReduceLROnPlateau` callback.

**Explanation:**
The bug is likely caused by the inconsistent handling of the `min_delta` attribute and the `iterations` attribute. The `min_delta` attribute is used to determine when to reduce the learning rate, while the `iterations` attribute is used to keep track of the number of iterations. The changes made in the bug patch may have introduced inconsistencies in how these attributes are handled, leading to unex

In [27]:
def build_rag_prompt(user_patch, rag_context):
    return f"""
You are a Python bug diagnosis assistant.

Your PRIMARY evidence is the USER BUG PATCH.

The retrieved historical bugs are only reference examples.
Do NOT assume that code, variables, classes, files, or problems
from the historical bugs exist in the user's bug.

Only mention a historical bug when its behavior is clearly relevant
to the user's patch.

If retrieved examples contain unrelated code, ignore them.

USER BUG PATCH:
{user_patch}

HISTORICAL BUG EXAMPLES:
{rag_context}

Analyze ONLY the user's patch and provide:

1. Likely root cause
2. Most likely bug category
3. Explanation based specifically on lines changed in the user patch
4. Recommended debugging steps
5. Suggested fix

For every root-cause claim, make sure it can be supported by
something visible in the USER BUG PATCH.
""".strip()

In [29]:
def build_rag_context(retrieved_bugs):
    context_parts = []

    for rank, (_, bug) in enumerate(retrieved_bugs.iterrows(), start=1):
        context_parts.append(
            f"""
HISTORICAL EXAMPLE {rank}

Category: {bug['bug_category']}
Reason: {bug['label_reason']}
Project: {bug['project']}
Modified Files: {bug['modified_files']}
Similarity: {bug['similarity_score']:.3f}
"""
        )

    return "\n".join(context_parts)

In [30]:
def build_rag_prompt(user_patch, rag_context):
    return f"""
You are analyzing a Python code patch for a bug.

STRICT RULES:

- Diagnose ONLY from code visible in USER PATCH.
- Historical examples are hints about possible bug categories.
- Never claim that a variable, method, class, file, or behavior exists
  in the user's code unless it appears in USER PATCH.
- Do not copy technical details from historical examples into the diagnosis.
- If evidence is insufficient, explicitly say so.
- Every root-cause claim must cite the relevant change visible in USER PATCH.

USER PATCH:
{user_patch}

RETRIEVED HISTORICAL EXAMPLES:
{rag_context}

Return:

1. Likely root cause
2. Most likely bug category
3. Evidence from the user's patch
4. Recommended debugging steps
5. Suggested fix
""".strip()

In [31]:
query_index = 0

results = retrieve_similar_bugs(
    bugs_df["patch"].iloc[query_index],
    bugs_df,
    historical_embeddings,
    embedding_model,
    tokenizer,
    top_k=3,
    exclude_index=query_index
)

results[
    [
        "project",
        "bug_id",
        "bug_category",
        "modified_files",
        "similarity_score"
    ]
]

,project,bug_id,bug_category,modified_files,similarity_score
20,keras,21,API and Configuration,['keras/callbacks.py'],0.741499
10,keras,29,Program Logic and State Management,['keras/engine/training.py'],0.632105
31,keras,4,API and Configuration,['keras/optimizers.py'],0.574832


In [32]:
rag_context = build_rag_context(results)

In [33]:
prompt = build_rag_prompt(
    bugs_df["patch"].iloc[query_index],
    rag_context
)

In [34]:
ollama_response = ollama.chat(

    model="llama3:8b",

    messages=[

        {

            "role": "user",

            "content": prompt

        }

    ]

)

ollama_answer = ollama_response["message"]["content"]

print(ollama_answer)

Based on the provided user patch and historical examples, here is the analysis:

**Likely root cause:**
The likely root cause is the change from `epsilon` to `min_delta` in the `ReduceLROnPlateau` class. This change may be causing unexpected behavior or errors in the code.

**Most likely bug category:**
The most likely bug category is "API and Configuration" since it involves changes to the callback's configuration options.

**Evidence from the user's patch:**
The evidence from the user's patch is the renaming of the `epsilon` parameter to `min_delta` in the `__init__` method and the subsequent changes to the `monitor_op` lambda function to use `min_delta` instead of `epsilon`.

**Recommended debugging steps:**
To debug this issue, I recommend:

1. Verifying that the `min_delta` parameter is being passed correctly to the `ReduceLROnPlateau` callback.
2. Checking the values of `min_delta` and `epsilon` (if they are still being used anywhere else in the code) to ensure they are within th

Generator

Result

GPT

Stronger reasoning; caught API change and state-management logic

Llama 3 8B

Grounded after prompt improvement, but missed part of the root cause

Original Llama prompt

Hallucinated unrelated retrieved-patch details

In [35]:
from sklearn.svm import LinearSVC
import joblib

final_classifier = LinearSVC(
    class_weight="balanced",
    random_state=42,
    max_iter=5000
)

final_classifier.fit(
    historical_embeddings,
    bugs_df["bug_category"]
)

joblib.dump(
    final_classifier,
    "../data/processed/minilm_svm_classifier.joblib"
)

print("Final classifier trained and saved.")

Final classifier trained and saved.


In [36]:
query_patch = bugs_df["patch"].iloc[query_index]

query_embedding = embed_patch(
    query_patch,
    embedding_model,
    tokenizer
)

predicted_category = final_classifier.predict(
    query_embedding.reshape(1, -1)
)[0]

print("Predicted category:", predicted_category)

Predicted category: API and Configuration


In [38]:
def build_rag_prompt(
    user_patch,
    predicted_category,
    rag_context
):
    return f"""
You are a Python bug diagnosis assistant.

Your PRIMARY evidence is the USER BUG PATCH.

The ML classifier prediction and retrieved historical bugs are
supporting signals only. They may be incorrect.

STRICT RULES:

- Diagnose only from code visible in USER PATCH.
- Do not invent variables, methods, classes, files, or behavior.
- Use the ML prediction as a hint, not as ground truth.
- Historical examples are references only.
- Ignore retrieved examples that are not relevant.
- Every root-cause claim must be supported by the USER PATCH.

USER BUG PATCH:
{user_patch}

ML CLASSIFIER SIGNAL:
Predicted bug category: {predicted_category}

RETRIEVED HISTORICAL EXAMPLES:
{rag_context}

Provide:

1. Likely root cause
2. Most likely bug category
3. Evidence from the user's patch
4. Recommended debugging steps
5. Suggested fix
""".strip()

In [39]:
prompt = build_rag_prompt(
    query_patch,
    predicted_category,
    rag_context
)

In [40]:
response = client.responses.create(
    model="gpt-5.6-luna",
    input=prompt
)

generated_answer = response.output_text

print(generated_answer)

1. **Likely root cause**

   The callback had two issues:

   - Its public threshold argument was named `epsilon`, although the intended API name is `min_delta`. This creates an API/configuration compatibility problem.
   - `wait` was incremented **after** checking `self.wait >= self.patience`. As a result, learning-rate reduction occurred one non-improving epoch later than the configured patience.

   For example, with `patience=10`, the old ordering checked the threshold while `wait` was still `9`, then incremented it to `10`; reduction happened on the following epoch.

2. **Most likely bug category**

   **Primary:** API and Configuration  
   **Secondary:** Program Logic and State Management

3. **Evidence from the patch**

   - The constructor changes:
     ```python
     epsilon=1e-4
     ```
     to:
     ```python
     min_delta=1e-4
     ```
   - Backward compatibility is added through:
     ```python
     if 'epsilon' in kwargs:
         min_delta = kwargs.pop('epsilon')
    

### End to End Pipeline

In [42]:
def generate_candidate_patch(
    buggy_code,
    problem_description,
    error_message=""
):
    prompt = f"""
You are a Python debugging assistant.

Analyze the following buggy Python code and problem description.

PROBLEM:
{problem_description}

ERROR / TRACEBACK:
{error_message}

BUGGY CODE:
{buggy_code}

Determine the likely root cause and propose a minimal fix.

Return exactly:

ROOT CAUSE:
<short explanation>

PROPOSED PATCH:
<unified-style diff showing only the necessary changes>
"""

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    return response.output_text

In [43]:
buggy_code = """
def get_user_age(user):
    return user["age"]
"""

problem_description = """
The application crashes when a user does not have an age field.
"""

error_message = """
KeyError: 'age'
"""

In [44]:
candidate_response = generate_candidate_patch(
    buggy_code,
    problem_description,
    error_message
)

print(candidate_response)

ROOT CAUSE:
`user["age"]` requires the key to exist, so it raises `KeyError` when the user has no age field.

PROPOSED PATCH:
```diff
 def get_user_age(user):
-    return user["age"]
+    return user.get("age")
```


In [45]:
def extract_patch(candidate_response):
    marker = "PROPOSED PATCH:"
    
    if marker not in candidate_response:
        return candidate_response

    patch = candidate_response.split(marker, 1)[1].strip()

    patch = patch.replace("```diff", "").replace("```", "").strip()

    return patch

In [46]:
candidate_patch = extract_patch(candidate_response)

print(candidate_patch)

def get_user_age(user):
-    return user["age"]
+    return user.get("age")


In [47]:
candidate_embedding = embed_patch(
    candidate_patch,
    embedding_model,
    tokenizer
)

In [48]:
candidate_category = final_classifier.predict(
    candidate_embedding.reshape(1, -1)
)[0]

print("Predicted category:", candidate_category)

Predicted category: API and Configuration


In [49]:
similar_bugs = retrieve_similar_bugs(
    candidate_patch,
    bugs_df,
    historical_embeddings,
    embedding_model,
    tokenizer,
    top_k=3
)

similar_bugs[
    [
        "project",
        "bug_id",
        "bug_category",
        "similarity_score"
    ]
]

,project,bug_id,bug_category,similarity_score
83,luigi,6,"File, I/O and Data Processing",0.343205
108,cookiecutter,3,API and Configuration,0.333922
143,scrapy,40,Type Handling,0.325561


In [50]:
rag_context = build_rag_context(similar_bugs)

In [51]:
def build_final_prompt(
    buggy_code,
    problem_description,
    error_message,
    candidate_patch,
    predicted_category,
    rag_context
):
    return f"""
You are a Python debugging assistant.

Analyze the original bug and evaluate the proposed fix.

The ML classifier prediction and retrieved historical bugs are
supporting signals only. They may be incorrect.

ORIGINAL BUGGY CODE:
{buggy_code}

PROBLEM DESCRIPTION:
{problem_description}

ERROR / TRACEBACK:
{error_message}

PROPOSED PATCH:
{candidate_patch}

ML CLASSIFIER SIGNAL:
Predicted bug category: {predicted_category}

SIMILAR HISTORICAL BUGS:
{rag_context}

Provide:

1. Root cause
2. Bug category
3. Explanation
4. Whether the proposed patch correctly fixes the bug
5. Recommended fix
6. Any edge cases or risks

Base your conclusions primarily on the original code,
problem description, error, and proposed patch.
Do not assume retrieved historical bugs have the same root cause.
""".strip()

In [52]:
final_prompt = build_final_prompt(
    buggy_code,
    problem_description,
    error_message,
    candidate_patch,
    candidate_category,
    rag_context
)

In [53]:
response = client.responses.create(
    model="gpt-5.6-luna",
    input=final_prompt
)

final_answer = response.output_text

print(final_answer)

1. **Root cause**  
   `get_user_age` uses `user["age"]`, which raises `KeyError` when the mapping does not contain an `"age"` key.

2. **Bug category**  
   Missing-field handling / input validation. This can also be considered a type/data-handling issue, rather than primarily an API or configuration problem.

3. **Explanation**  
   Direct dictionary indexing assumes that `"age"` is always present. The problem states that users may legitimately lack this field, so the function must define what “unknown age” means instead of allowing a `KeyError` to propagate.

4. **Does the proposed patch fix the bug?**  
   Yes, for the stated failure:

   ```python
   def get_user_age(user):
       return user.get("age")
   ```

   `dict.get("age")` returns `None` when the key is absent, preventing the `KeyError`. It also preserves valid falsy values such as `0`.

   However, this is correct only if `None` is an acceptable representation of an unknown age and callers can handle it.

5. **Recommende